# Unified MoE-FFD Kaggle pipeline: FF++ C23 download → face crops → training → evaluation

Run this notebook **top to bottom** in a Kaggle Notebook with **Internet = On** and **GPU = On**.

This merges the two supplied notebooks into one workflow:
1. Obtain the official FF++ C23 videos using the official FaceForensics++ download script supplied as a Kaggle Input (or use an already-downloaded raw-video folder).
2. Use the original preprocessing implementation: **20 uniformly sampled frames per video, MTCNN face detection, largest-face crop, 0.30 face-box margin, JPEG quality 95**.
3. Save crops in the MoE-FFD folder structure and optionally package/upload each completed stage.
4. Clone MoE-FFD, apply and verify the Multi-Pool head patch, train the modified model for the same 5 epochs / batch size 32 / seed 2024, load the supplied baseline checkpoint, and evaluate both models.
5. Run final integrity and smoke checks.

**Important external prerequisites:** FF++ is access-controlled. You must have accepted the dataset terms and add the official FF++ `download-FaceForensics.py` script (or the folder containing it) as a Kaggle Input. The notebook cannot bypass the dataset access approval. For Celeb-DF evaluation, attach its source ZIP/folder and provide its path in the settings cell; it is optional and is not downloaded by this notebook.

**Expected output structure:** `/kaggle/working/data/c23/{train,valid,test}/{original,Deepfakes,Face2Face,FaceSwap,NueralTextures}/<video>/<frame>.jpg`

In [ ]:
# ======================== EDIT ONLY THIS CELL ========================
# FaceForensics++ raw-video source:
# Leave FF_VIDEO_ROOT="" to use the official download script attached as Kaggle Input.
# Or set it to a folder containing original_sequences/ and manipulated_sequences/.
FF_VIDEO_ROOT = ""

# Optional Celeb-DF v2 ZIP or extracted directory (must include List_of_testing_videos.txt).
# Leave empty to skip Celeb-DF.
CDF_SOURCE = ""

# Stages from the supplied preparation notebook. Keep the names exactly as written.
STAGES = ["original", "Deepfakes", "Face2Face", "FaceSwap", "NueralTextures"]

# Preserve the original crop specifications from your preparation notebook.
N_FRAMES = 20
MARGIN = 0.30
SERVER = "EU2"
SAVE_PREFIX = "ffpp-c23-crops"

# Model/training settings from the supplied training notebook.
TEAM_REPO = "https://github.com/yagnaveenaaa/MoE-FFD.git"
NUM_WORKERS = 4
EPOCHS = 5
BATCH_SIZE = 32
SEED = 2024

# ======================== DO NOT EDIT BELOW ==========================
import os, sys, glob, json, time, shutil, subprocess, hashlib, tarfile
WORK = "/kaggle/working"
RAW_DL = f"{WORK}/ffraw"
CDF_RAW = f"{WORK}/cdfraw"
DATA = f"{WORK}/data"
DATA_ROOT = f"{DATA}/c23"
CDF_ROOT = f"{DATA}/CDF" if CDF_SOURCE else ""
PROG = f"{WORK}/progress"
FLAGS = f"{WORK}/flags"
EXP = f"{WORK}/export"
REPO = f"{WORK}/team"
PREP_LIB = f"{WORK}/prep_lib.py"
PREP_WORKER = f"{WORK}/prep_worker.py"
SCRIPT = f"{WORK}/moe_ffd_multipool.py"
OUT = f"{WORK}/models/multipool"
BASE_CKPT = f"{WORK}/models/baseline/models_params_4.tar"
MP_CKPT = f"{OUT}/models_params_{EPOCHS-1}.tar"
RES = f"{WORK}/results"
for d in (DATA, PROG, FLAGS, EXP, RES, os.path.dirname(BASE_CKPT)):
    os.makedirs(d, exist_ok=True)
if CDF_SOURCE:
    STAGES = ["CDF"] + STAGES
print("Stages:", STAGES)
print("Crop settings:", {"frames_per_video": N_FRAMES, "MTCNN": True, "largest_face_only": True, "margin": MARGIN, "jpeg_quality": 95})
print("Training settings:", {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "seed": SEED, "num_workers": NUM_WORKERS})


## 1. Kaggle environment and dependencies

In [ ]:
import subprocess, shutil, importlib, torch
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "GPU not detected; enable GPU in Notebook Settings.")
print("Free disk (GB):", round(shutil.disk_usage(WORK).free / 1e9, 1))
try:
    from facenet_pytorch import MTCNN
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "facenet-pytorch", "--no-deps"])
    from facenet_pytorch import MTCNN
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
print("MTCNN import: OK")


In [ ]:
%%writefile /kaggle/working/prep_lib.py
"""Helpers for the raw-video -> cropped-face-frames notebook (FF++ C23 and Celeb-DF test)."""
import glob, json, os, re, shutil, subprocess, tarfile, time, zipfile

# output folder names (the repo's dataset.py needs 'original' exactly; it spells NeuralTextures 'NueralTextures')
FF_STAGES = ["original", "Deepfakes", "Face2Face", "FaceSwap", "NueralTextures"]
RAW_SUBDIR = {
    "original": "original_sequences/youtube",
    "Deepfakes": "manipulated_sequences/Deepfakes",
    "Face2Face": "manipulated_sequences/Face2Face",
    "FaceSwap": "manipulated_sequences/FaceSwap",
    "NueralTextures": "manipulated_sequences/NeuralTextures",
}
DL_NAME = {"original": "original", "Deepfakes": "Deepfakes", "Face2Face": "Face2Face",
           "FaceSwap": "FaceSwap", "NueralTextures": "NeuralTextures"}
SPLIT_DIR = {"train": "train", "val": "valid", "test": "test"}
EXPECTED = {"train": 720, "valid": 140, "test": 140}   # official FF++ split, videos per method
TAR_NAME = {s: "ffpp_c23_%s.tar" % s for s in FF_STAGES}
TAR_NAME["CDF"] = "cdf_test.tar"


# ------------------------------------------------------------------ splits and jobs
def load_splits(path):
    return json.load(open(path))


def id_to_split(splits):
    sid = {}
    for s, pairs in splits.items():
        for a, b in pairs:
            sid[a] = s
            sid[b] = s
    return sid


def ff_video_names(splits, stage):
    ids = [v for pairs in splits.values() for pair in pairs for v in pair]
    if stage == "original":
        return sorted(set(ids))
    names = []
    for pairs in splits.values():
        for a, b in pairs:
            names += ["%s_%s" % (a, b), "%s_%s" % (b, a)]
    return sorted(names)


def build_ff_jobs(splits, stages, raw_root, data_root, flag_dir=None):
    """One job per video. raw_root contains original_sequences/ and manipulated_sequences/."""
    sid = id_to_split(splits)
    jobs = []
    for stage in stages:
        for vid in ff_video_names(splits, stage):
            sp = SPLIT_DIR[sid[vid.split("_")[0]]]
            assert all(SPLIT_DIR[sid[x]] == sp for x in vid.split("_")), "split mismatch " + vid
            jobs.append({
                "stage": stage, "vid": vid,
                "src": "%s/%s/c23/videos/%s.mp4" % (raw_root, RAW_SUBDIR[stage], vid),
                "out": "%s/c23/%s/%s/%s" % (data_root, sp, stage, vid),
                "flag": ("%s/dl_%s.done" % (flag_dir, stage)) if flag_dir else "",
                "failflag": ("%s/dl_%s.failed" % (flag_dir, stage)) if flag_dir else "",
            })
    return jobs


def find_ff_raw_root(base):
    """Locate the folder that contains original_sequences/ and manipulated_sequences/."""
    hits = glob.glob(os.path.join(base, "**", "original_sequences"), recursive=True)
    if not hits:
        return None
    return os.path.dirname(hits[0])


# ------------------------------------------------------------------ progress
def read_progress(progress_dir):
    out = {}
    for f in glob.glob(os.path.join(progress_dir, "p_*.tsv")):
        for line in open(f):
            parts = line.rstrip("\n").split("\t")
            if len(parts) == 3:
                out.setdefault(parts[0], {})[parts[1]] = parts[2]
    return out


def stage_status(jobs, progress):
    """stage -> dict(total, done, ok, noface, missing, error)"""
    res = {}
    for j in jobs:
        r = res.setdefault(j["stage"], {"total": 0, "done": 0, "ok": 0, "noface": 0, "missing": 0, "error": 0})
        r["total"] += 1
        st = progress.get(j["stage"], {}).get(j["vid"])
        if st:
            r["done"] += 1
            r[st if st in r else "error"] += 1
    return res


# ------------------------------------------------------------------ packaging and saving
def package_stage(stage, data_root, export_dir):
    """Tar all finished folders of a stage (arcnames relative to data_root). Returns tar path."""
    os.makedirs(export_dir, exist_ok=True)
    tar_path = os.path.join(export_dir, TAR_NAME[stage])
    if stage == "CDF":
        dirs = [os.path.join(data_root, "CDF")]
    else:
        dirs = sorted(glob.glob(os.path.join(data_root, "c23", "*", stage)))
    dirs = [d for d in dirs if os.path.isdir(d)]
    if not dirs:
        raise RuntimeError("nothing to package for " + stage)
    tmp = tar_path + ".tmp"
    with tarfile.open(tmp, "w") as tf:
        for d in dirs:
            tf.add(d, arcname=os.path.relpath(d, data_root))
    os.replace(tmp, tar_path)
    return tar_path


def slug(stage):
    return {"CDF": "cdf-test", "NueralTextures": "neuraltextures"}.get(stage, stage.lower())


def push_stage(stage, tar_path, user, prefix, export_dir, env):
    """Create (or version) a private Kaggle dataset that holds this stage's tar. Returns (ok, message)."""
    ds_slug = "%s-%s" % (prefix, slug(stage))
    d = os.path.join(export_dir, "ds_" + slug(stage))
    os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.basename(tar_path))
    if not os.path.exists(dst):
        try:
            os.link(tar_path, dst)
        except OSError:
            shutil.copy(tar_path, dst)
    meta = {"title": ("%s %s" % (prefix.replace("-", " "), slug(stage).replace("-", " ")))[:50].ljust(6, "."),
            "id": "%s/%s" % (user, ds_slug), "licenses": [{"name": "other"}]}
    json.dump(meta, open(os.path.join(d, "dataset-metadata.json"), "w"))
    r = subprocess.run(["kaggle", "datasets", "create", "-p", d], env=env, capture_output=True, text=True)
    msg = (r.stdout + r.stderr).strip()
    if r.returncode == 0 and "error" not in msg.lower():
        return True, "created private dataset %s" % meta["id"]
    if "already" in msg.lower() or "exist" in msg.lower():
        r2 = subprocess.run(["kaggle", "datasets", "version", "-p", d, "-m", "update " + stage],
                            env=env, capture_output=True, text=True)
        m2 = (r2.stdout + r2.stderr).strip()
        return (r2.returncode == 0 and "error" not in m2.lower()), "new version of %s: %s" % (meta["id"], m2[-300:])
    return False, msg[-400:]


def glob_depth(root, pattern, max_depth=4):
    """Like a recursive glob but only a few levels deep (fast on big /kaggle/input trees)."""
    hits = []
    for d in range(0, max_depth + 1):
        hits += glob.glob(os.path.join(root, *(["*"] * d), pattern))
    return hits


def extract_saved_tars(input_root, data_root):
    """Resume support: unpack tars found in attached inputs. Returns the list of stages recovered."""
    got = []
    for stage, name in TAR_NAME.items():
        hits = glob_depth(input_root, name)
        if hits:
            with tarfile.open(hits[0]) as tf:
                tf.extractall(data_root)
            got.append(stage)
    return got


# ------------------------------------------------------------------ Celeb-DF
def _tail2(path):
    parts = path.replace("\\", "/").strip("/").split("/")
    return "/".join(parts[-2:])


def cdf_collect(source):
    """Returns (list_text or None, {relpath 'Folder/name.mp4': ('zip', zippath, member) | ('file', path)})."""
    vids, list_text = {}, None
    if os.path.isfile(source) and zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as zf:
            for n in zf.namelist():
                if n.lower().endswith("list_of_testing_videos.txt"):
                    list_text = zf.read(n).decode("utf-8", "ignore")
                elif n.lower().endswith(".mp4"):
                    vids[_tail2(n)] = ("zip", source, n)
    elif os.path.isdir(source):
        for p in glob.glob(os.path.join(source, "**", "*"), recursive=True):
            if p.lower().endswith("list_of_testing_videos.txt"):
                list_text = open(p, encoding="utf-8", errors="ignore").read()
            elif p.lower().endswith(".mp4"):
                vids[_tail2(p)] = ("file", p)
    else:
        raise FileNotFoundError("CDF source is neither a zip nor a folder: %s" % source)
    return list_text, vids


def cdf_test_entries(list_text, vids):
    """Official test videos (by List_of_testing_videos.txt) -> list of relpaths that exist in vids."""
    want = []
    for line in list_text.splitlines():
        line = line.strip()
        if line:
            want.append(_tail2(line.split()[-1]))
    present = [w for w in want if w in vids]
    return want, present


def cdf_materialize(entries, vids, raw_dir):
    """Extract the chosen videos from the zip into raw_dir (folders are used in place). relpath -> local path."""
    out = {}
    zf_cache = {}
    for rel in entries:
        kind = vids[rel]
        if kind[0] == "file":
            out[rel] = kind[1]
            continue
        dst = os.path.join(raw_dir, rel)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            zf = zf_cache.setdefault(kind[1], zipfile.ZipFile(kind[1]))
            with zf.open(kind[2]) as src, open(dst + ".part", "wb") as f:
                shutil.copyfileobj(src, f)
            os.replace(dst + ".part", dst)
        out[rel] = dst
    for z in zf_cache.values():
        z.close()
    return out


def build_cdf_jobs(local_paths, data_root):
    jobs = []
    for rel, p in sorted(local_paths.items()):
        folder, name = rel.split("/")
        vid = os.path.splitext(name)[0]
        jobs.append({"stage": "CDF", "vid": "%s/%s" % (folder, vid), "src": p,
                     "out": "%s/CDF/%s/%s" % (data_root, folder, vid), "flag": "", "failflag": ""})
    return jobs


In [ ]:
%%writefile /kaggle/working/prep_worker.py
"""Crop faces from videos. Usage: python prep_worker.py JOBS.json SHARD NSHARD DEVICE PROGRESS_DIR N_FRAMES MARGIN"""
import json, os, shutil, sys, time
import cv2, numpy as np
from PIL import Image
from facenet_pytorch import MTCNN

jobs_path, shard, nshard, dev, prog_dir = sys.argv[1], int(sys.argv[2]), int(sys.argv[3]), sys.argv[4], sys.argv[5]
N_FRAMES, MARGIN, CHUNK = int(sys.argv[6]), float(sys.argv[7]), 10
os.makedirs(prog_dir, exist_ok=True)
mtcnn = MTCNN(keep_all=True, device=dev)


def crop_video(path, out_dir):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return -1
    want = set(np.linspace(0, max(total - 1, 0), N_FRAMES).astype(int).tolist())
    frames, i, last = [], 0, max(want)
    while i <= last:
        if not cap.grab():
            break
        if i in want:
            ok, fr = cap.retrieve()
            if ok:
                frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        i += 1
    cap.release()
    tmp = out_dir + "_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    saved = 0
    for c in range(0, len(frames), CHUNK):
        chunk = frames[c:c + CHUNK]
        boxes, _ = mtcnn.detect(chunk)
        for rgb, bx in zip(chunk, boxes):
            if bx is None:
                continue
            x1, y1, x2, y2 = max(bx, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
            m = MARGIN * max(x2 - x1, y2 - y1)
            h, w = rgb.shape[:2]
            x1, y1, x2, y2 = int(max(0, x1 - m)), int(max(0, y1 - m)), int(min(w, x2 + m)), int(min(h, y2 + m))
            Image.fromarray(rgb[y1:y2, x1:x2]).save("%s/%d.jpg" % (tmp, saved), quality=95)
            saved += 1
    if saved == 0:
        shutil.rmtree(tmp, ignore_errors=True)
        return 0
    os.replace(tmp, out_dir)
    return saved


jobs = [j for k, j in enumerate(json.load(open(jobs_path))) if k % nshard == shard]
prog = open(os.path.join(prog_dir, "p_%d.tsv" % shard), "a")
few = open(os.path.join(prog_dir, "few_%d.log" % shard), "a")


def mark(j, status):
    prog.write("%s\t%s\t%s\n" % (j["stage"], j["vid"], status)); prog.flush()


for j in jobs:
    if os.path.isdir(j["out"]):
        mark(j, "ok"); continue
    t0 = time.time()
    while j["flag"] and not os.path.exists(j["flag"]):       # wait until this stage's download finished
        if j["failflag"] and os.path.exists(j["failflag"]):
            break
        time.sleep(10)
        if time.time() - t0 > 4 * 3600:
            break
    if not os.path.exists(j["src"]):
        mark(j, "missing"); continue
    os.makedirs(os.path.dirname(j["out"]), exist_ok=True)
    try:
        n = crop_video(j["src"], j["out"])
    except Exception as e:
        few.write("ERROR %s %s %r\n" % (j["stage"], j["vid"], e)); few.flush()
        shutil.rmtree(j["out"] + "_tmp", ignore_errors=True)
        mark(j, "error"); continue
    if n > 0:
        if n < N_FRAMES:
            few.write("FEW %s %s %d\n" % (j["stage"], j["vid"], n)); few.flush()
        mark(j, "ok")
    else:
        mark(j, "noface" if n == 0 else "error")
open(os.path.join(prog_dir, "done_%d.flag" % shard), "w").write("done")


## 2. Optional automatic saving to a private Kaggle Dataset

In [ ]:
import sys; sys.path.insert(0, WORK)
import prep_lib as L
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = sec.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = sec.get_secret("KAGGLE_KEY")
except Exception as e:
    print("Secrets not found:", repr(e)[:120])
USER = os.environ.get("KAGGLE_USERNAME", "")
SAVING = bool(USER and os.environ.get("KAGGLE_KEY"))
if SAVING:
    r = subprocess.run(["kaggle", "datasets", "list", "-m"], capture_output=True, text=True)
    SAVING = (r.returncode == 0)
    print("Kaggle API check:", "OK" if SAVING else ("FAILED: " + (r.stdout + r.stderr)[-300:]))
print("AUTOMATIC SAVING:", "ENABLED as " + USER if SAVING else "DISABLED - results will only stay in this session (use Save Version at the end)")

If automatic saving is disabled, the pipeline still writes tar archives under `/kaggle/working/export`; use **Save Version** to preserve notebook outputs. Kaggle API credentials are only needed for automatic dataset publishing.

## 3. Load official FF++ split lists and recover any previously saved stage archives

In [ ]:
import urllib.request
base = "https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/"
splits = {s: json.load(urllib.request.urlopen(base + s + ".json")) for s in ["train", "val", "test"]}
json.dump(splits, open(f"{WORK}/splits.json", "w"))
ids = [v for p in splits.values() for pair in p for v in pair]
print({k: len(v) for k, v in splits.items()}, "| unique ids:", len(set(ids)), "(expected 360/70/70 pairs, 1000 ids)")
assert len(set(ids)) == 1000, "unexpected split lists - stop and send this output"

In [ ]:
recovered = L.extract_saved_tars("/kaggle/input", DATA)
for s in recovered:
    open(f"{EXP}/packaged_{s}.flag", "w").write("x"); open(f"{EXP}/pushed_{s}.flag", "w").write("x")
print("recovered from saved datasets:", recovered or "nothing (fresh start)")
TODO = [s for s in STAGES if s not in recovered]
print("stages still to build:", TODO)

## 4. Obtain the raw FF++ videos

When `FF_VIDEO_ROOT` is empty, this section searches Kaggle Inputs for the official `download-FaceForensics.py` script. The downloader uses the `EU2` server and C23 video compression, as in the supplied preparation notebook. If you already attached the extracted raw videos, set `FF_VIDEO_ROOT` in the settings cell instead.

In [ ]:
FF_NEEDED = [s for s in TODO if s != "CDF"]
RAW_ROOT, FLAG_DIR = None, None
if FF_NEEDED:
    if FF_VIDEO_ROOT:
        RAW_ROOT = L.find_ff_raw_root(FF_VIDEO_ROOT)
        assert RAW_ROOT, "original_sequences/ not found under FF_VIDEO_ROOT - list that folder and send the listing"
        print("using existing videos at", RAW_ROOT)
    else:
        def good(p):
            o = subprocess.run(["python", p, "-h"], capture_output=True, text=True)
            t = o.stdout + o.stderr
            return "--server" in t and "--compression" in t
        cands = [p for p in L.glob_depth("/kaggle/input", "*.py") if good(p)]
        assert cands, "FaceForensics++ download script (the one with --server/--compression) is not attached as Input"
        FF_DOWNLOAD_SCRIPT = cands[0]; RAW_ROOT = RAW_DL; FLAG_DIR = FLAGS
        assert subprocess.run(["pgrep", "-f", "dl_ff.sh"], capture_output=True).returncode != 0, "download already running"
        for f in glob.glob(f"{FLAGS}/dl_*"): os.remove(f)
        sh = "#!/bin/bash\n"
        for st in FF_NEEDED:
            sh += (f"ok=0\nfor i in 1 2 3; do\n"
                   f'  echo "" | python {FF_DOWNLOAD_SCRIPT} {RAW_ROOT} -d {L.DL_NAME[st]} -c c23 -t videos --server {SERVER} >> {WORK}/dl_{st}.log 2>&1 && ok=1 && break\n'
                   f"  sleep 30\ndone\n"
                   f"if [ $ok = 1 ]; then touch {FLAGS}/dl_{st}.done; else touch {FLAGS}/dl_{st}.failed; fi\n")
        open(f"{WORK}/dl_ff.sh", "w").write(sh)
        subprocess.Popen(f"nohup bash {WORK}/dl_ff.sh > {WORK}/dl_ff.out 2>&1 &", shell=True)
        print("download started in the background with script:", FF_DOWNLOAD_SCRIPT)
else:
    print("no FF++ stage needed")

## 5. Optional Celeb-DF official test set

In [ ]:
cdf_jobs = []
if "CDF" in TODO:
    list_text, vids = L.cdf_collect(CDF_SOURCE)
    assert list_text, ("List_of_testing_videos.txt not found in CDF_SOURCE. It defines the official test set, so do not guess: "
                       "check the zip/folder listing and send it to the team.")
    want, present = L.cdf_test_entries(list_text, vids)
    print("videos in source:", len(vids), "| listed as test:", len(want), "| found:", len(present),
          "(the official Celeb-DF v2 test list has 518 entries, as I recall - compare with your list)")
    assert present, "no listed test video was found in the source"
    if len(present) != len(want):
        print("WARNING: missing from source:", sorted(set(want) - set(present))[:10])
    local = L.cdf_materialize(present, vids, CDF_RAW)
    cdf_jobs = L.build_cdf_jobs(local, DATA)
    from collections import Counter
    print("test videos per folder:", Counter(j["vid"].split("/")[0] for j in cdf_jobs))

## 6. Crop videos into face frames

The worker preserves the original implementation: sample 20 frames uniformly across each video; run MTCNN; keep the largest detected face per frame; expand the bounding box by 0.30 × the larger face dimension; clip to image boundaries; save sequential JPEGs at quality 95. A video with no detected face is recorded as `noface`, not silently counted as a successful crop.

In [ ]:
assert subprocess.run(["pgrep", "-f", "prep_worker.py"], capture_output=True).returncode != 0, "workers already running - do not start twice"
assert shutil.disk_usage(WORK).free > 4e9, "less than 4 GB free"
ff_jobs = L.build_ff_jobs(splits, FF_NEEDED, RAW_ROOT, DATA, FLAG_DIR) if FF_NEEDED else []
jobs = cdf_jobs + ff_jobs
json.dump(jobs, open(f"{WORK}/jobs.json", "w"))
for d in glob.glob(f"{DATA}/**/*_tmp", recursive=True): shutil.rmtree(d, ignore_errors=True)      # leftovers of a crashed run
for f in glob.glob(f"{PROG}/done_*.flag"): os.remove(f)
NW = max(1, NGPU)
for k in range(NW):
    dev = f"cuda:{k}" if NGPU else "cpu"
    subprocess.Popen(f"nohup python {WORK}/prep_worker.py {WORK}/jobs.json {k} {NW} {dev} {PROG} {N_FRAMES} {MARGIN} > {WORK}/worker_{k}.out 2>&1 &", shell=True)
print("jobs:", len(jobs), "| workers:", NW, "| device:", "cuda" if NGPU else "cpu")

In [ ]:
import time
order = []
for j in jobs:
    if j["stage"] not in order: order.append(j["stage"])
tries = {}
def flag(kind, st): return f"{EXP}/{kind}_{st}.flag"
t_start, last = time.time(), None
while True:
    st = L.stage_status(jobs, L.read_progress(PROG))
    for s in order:
        x = st.get(s)
        if not x or x["done"] < x["total"]: continue
        if not os.path.exists(flag("packaged", s)):
            if x["missing"] or x["error"]:
                print(f"WARNING {s}: missing={x['missing']} error={x['error']} (packaging what exists)")
            tar = L.package_stage(s, DATA, EXP); open(flag("packaged", s), "w").write(tar)
            print(f"[packaged] {s}: {os.path.getsize(tar)/1e6:.0f} MB")
            raw = f"{CDF_RAW}" if s == "CDF" else (f"{RAW_DL}/{L.RAW_SUBDIR[s]}" if RAW_ROOT == RAW_DL else None)
            if raw and os.path.isdir(raw): shutil.rmtree(raw, ignore_errors=True); print(f"[freed] raw videos of {s}")
        if SAVING and not os.path.exists(flag("pushed", s)) and tries.get(s, 0) < 3:
            tries[s] = tries.get(s, 0) + 1
            ok, msg = L.push_stage(s, open(flag("packaged", s)).read(), USER, SAVE_PREFIX, EXP, dict(os.environ))
            print(f"[saved] {s}: {'OK' if ok else 'FAILED'} - {msg[:160]}")
            if ok: open(flag("pushed", s), "w").write("x")
    line = " | ".join(f"{s}: {st[s]['done']}/{st[s]['total']}" for s in order if s in st)
    if line != last: print(time.strftime("%H:%M:%S"), line); last = line
    all_done = all(st.get(s) and st[s]["done"] == st[s]["total"] for s in order)
    saved_all = all(os.path.exists(flag("packaged", s)) and (not SAVING or os.path.exists(flag("pushed", s)) or tries.get(s, 0) >= 3) for s in order)
    if all_done and saved_all: break
    time.sleep(60)
print("ALL STAGES FINISHED in %.1f min" % ((time.time() - t_start) / 60))

## 7. Verify the cropped dataset before training

In [ ]:
st = L.stage_status(jobs, L.read_progress(PROG))
print("status per stage:", json.dumps(st, indent=1))
rows = []
for sp in ["train", "valid", "test"]:
    row = {m: len([d for d in glob.glob(f"{DATA}/c23/{sp}/{m}/*") if os.path.isdir(d)]) for m in L.FF_STAGES if m in STAGES}
    if row: print(sp, row, "| expected per method:", L.EXPECTED[sp])
if "CDF" in STAGES:
    print("CDF:", {f: len(os.listdir(f"{DATA}/CDF/{f}")) for f in os.listdir(f"{DATA}/CDF")})
print("videos with fewer than %d frames / errors:" % N_FRAMES)
print(subprocess.run(f"cat {PROG}/few_*.log | head -15; echo total lines: $(cat {PROG}/few_*.log | wc -l)", shell=True, capture_output=True, text=True).stdout)
print("saved flags:", sorted(os.path.basename(f) for f in glob.glob(f"{EXP}/pushed_*.flag")))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
picks = []
for s in order:
    ds = sorted(glob.glob(f"{DATA}/CDF/*/*")) if s == "CDF" else sorted(glob.glob(f"{DATA}/c23/train/{s}/*"))
    if ds: picks.append((s, ds[0]))
fig, ax = plt.subplots(1, len(picks), figsize=(3 * len(picks), 3))
ax = [ax] if len(picks) == 1 else ax
for a, (s, d) in zip(ax, picks):
    a.imshow(Image.open(f"{d}/0.jpg")); a.set_title(s, fontsize=9); a.axis("off")
plt.show()

**Stop if** the sample crops do not show one correctly cropped face, or if there are substantial `missing` / `error` counts. A few `noface` videos can occur, but they reduce the usable dataset. Training will only start after the folder structure is checked.

In [ ]:
# Fail-fast structural checks for the five FF++ C23 classes.
METHODS = ["original", "Deepfakes", "Face2Face", "FaceSwap", "NueralTextures"]
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(DATA_ROOT, split)
    assert os.path.isdir(split_dir), f"Missing split folder: {split_dir}"
    for method in METHODS:
        method_dir = os.path.join(split_dir, method)
        assert os.path.isdir(method_dir), f"Missing method folder: {method_dir}"
        videos = [p for p in glob.glob(os.path.join(method_dir, "*")) if os.path.isdir(p)]
        assert videos, f"No cropped video folders found in {method_dir}"
        assert any(glob.glob(os.path.join(v, "*.jpg")) for v in videos), f"No JPEG crops found in {method_dir}"
print("DATA STRUCTURE SMOKE TEST: PASS")


## 8. Clone the MoE-FFD repository

In [ ]:
if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-q", TEAM_REPO, REPO])
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--format=commit %h  %s"], capture_output=True, text=True, check=True).stdout)
req = os.path.join(REPO, "requirements.txt")
if os.path.isfile(req):
    print(open(req).read())


## 9. Check required training packages

In [ ]:
# Install only missing non-Torch dependencies; never upgrade the Kaggle-provided torch stack.
import importlib.util
packages = {
    "timm": "timm",
    "albumentations": "albumentations",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "cv2": "opencv-python-headless",
    "tqdm": "tqdm",
}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
for module in packages:
    imported = __import__(module)
    print(module, getattr(imported, "__version__", "available"))
import torch
print("torch", torch.__version__, "(left unchanged)")


## 10. Write the Multi-Pool training/evaluation helper

In [ ]:
%%writefile /kaggle/working/moe_ffd_multipool.py
#!/usr/bin/env python3
"""
MoE-FFD baseline + Multi-Pool Head novelty  (one script, one switch)
=====================================================================

WHAT IT DOES
  The authors' MoE-FFD classifier reads only the CLS token.  The novelty reads
        z = [ CLS ; mean(patch tokens) ; max(patch tokens) ]      (768 -> 2304 inputs)
  Everything else (data, 5 epochs, seed 2024, batch 32, learning rates, losses,
  evaluation) stays identical to the baseline.

  The switch is the environment variable HEAD_MODE, read by ViT_MoE.py:
      HEAD_MODE=cls         -> original baseline model   (default if unset)
      HEAD_MODE=multipool   -> the novelty               (proposed model)
      HEAD_MODE=clsmean / clsmax -> optional ablations

  IMPORTANT: a checkpoint must always be loaded with the SAME HEAD_MODE it was
  trained with.  The `train` and `eval` commands below set it for you.

COMMANDS  (run from anywhere; --repo is the team MoE-FFD checkout)
  1) python moe_ffd_multipool.py patch   --repo REPO
  2) python moe_ffd_multipool.py check   --repo REPO          # parameter counts
  3) python moe_ffd_multipool.py train   --repo REPO --data_root DATA/c23 --out OUTDIR \
                                         --num_workers N       # N = same as baseline run
  4) python moe_ffd_multipool.py eval    --repo REPO --ckpt OUTDIR/models_params_4.tar \
                                         --test_root DATA/c23/test --dataset FFPP --out_json r.json
  5) python moe_ffd_multipool.py compare --a baseline.json --b multipool.json
  (undo the patch any time:  python moe_ffd_multipool.py unpatch --repo REPO)

EXPECTED TRAINABLE PARAMETERS (hand-computed from the code; `check` verifies them)
  cls (baseline) 15,510,722   |   multipool 15,513,794 (+3,072)
"""
import argparse
import json
import os
import shlex
import subprocess
import sys
import time

MARKER = "# --- MULTIPOOL_HEAD_PATCH ---"
EXPECTED = {
    "cls": 15510722,
    "multipool": 15513794,
    "clsmean": 15512258,
    "clsmax": 15512258,
}
HEAD_IN = {"cls": 768, "multipool": 2304, "clsmean": 1536, "clsmax": 1536}

ANCHOR_LOG = "_logger = logging.getLogger(__name__)"
ANCHOR_HEAD = ("self.head = nn.Linear(self.num_features, num_classes) "
               "if num_classes > 0 else nn.Identity()")
ANCHOR_RET = "return self.pre_logits(x[:, 0]),moe_loss"

HELPER = '''

%s
import os as _os
HEAD_MODE = _os.environ.get('HEAD_MODE', 'cls')   # cls (baseline) | multipool | clsmean | clsmax
_HEAD_MULTS = {'cls': 1, 'multipool': 3, 'clsmean': 2, 'clsmax': 2}
if HEAD_MODE not in _HEAD_MULTS:
    raise ValueError('HEAD_MODE must be one of %%s, got %%r' %% (list(_HEAD_MULTS), HEAD_MODE))
_HEAD_MULT = _HEAD_MULTS[HEAD_MODE]

def pool_tokens(x):
    """x: [B, 1+N, C] after the final norm. Returns the feature vector fed to the head."""
    cls, pt = x[:, 0], x[:, 1:]
    if HEAD_MODE == 'cls':
        return cls
    if HEAD_MODE == 'multipool':
        return torch.cat([cls, pt.mean(1), pt.max(1).values], dim=1)
    if HEAD_MODE == 'clsmean':
        return torch.cat([cls, pt.mean(1)], dim=1)
    return torch.cat([cls, pt.max(1).values], dim=1)
''' % MARKER


# ----------------------------------------------------------------------------- helpers
def vit_path(repo):
    return os.path.join(repo, "ViT_MoE.py")


def is_patched(repo):
    return MARKER in open(vit_path(repo)).read()


def run_tee(cmd, env, cwd, log_path):
    """Run a command, echo its output and append it to log_path. Returns exit code."""
    print("RUN:", " ".join(shlex.quote(c) for c in cmd), flush=True)
    with open(log_path, "a") as lf:
        p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line)
            lf.write(line)
        return p.wait()


def git_hash(repo):
    try:
        return subprocess.check_output(["git", "-C", repo, "rev-parse", "HEAD"],
                                       text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unknown"


def lib_versions():
    code = ("import torch, timm; print(torch.__version__, timm.__version__)")
    try:
        return subprocess.check_output([sys.executable, "-c", code], text=True,
                                       stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unknown"


# ----------------------------------------------------------------------------- commands
def cmd_patch(a):
    p = vit_path(a.repo)
    if not os.path.isfile(p):
        sys.exit("ViT_MoE.py not found in %s" % a.repo)
    s = open(p).read()
    if MARKER in s:
        print("Already patched - nothing to do.")
        return
    missing = [n for n, t in (("log anchor", ANCHOR_LOG), ("head line", ANCHOR_HEAD),
                              ("return line", ANCHOR_RET)) if s.count(t) != 1]
    if missing:
        sys.exit("ViT_MoE.py differs from the authors' version (cannot find exactly one of: %s). "
                 "Nothing was changed. Send the file/lines to the person who wrote this script."
                 % ", ".join(missing))
    if not os.path.exists(p + ".orig"):
        open(p + ".orig", "w").write(s)
    s = s.replace(ANCHOR_LOG, ANCHOR_LOG + HELPER)
    s = s.replace(ANCHOR_HEAD,
                  "self.head = nn.Linear(self.num_features * _HEAD_MULT, num_classes) "
                  "if num_classes > 0 else nn.Identity()")
    s = s.replace(ANCHOR_RET, "return self.pre_logits(pool_tokens(x)),moe_loss")
    open(p, "w").write(s)
    print("Patched %s (backup: ViT_MoE.py.orig). Default HEAD_MODE=cls reproduces the baseline." % p)


def cmd_unpatch(a):
    p = vit_path(a.repo)
    if not os.path.exists(p + ".orig"):
        sys.exit("No backup ViT_MoE.py.orig found - nothing to restore.")
    open(p, "w").write(open(p + ".orig").read())
    print("Restored the original ViT_MoE.py")


def cmd_check(a):
    if not is_patched(a.repo):
        sys.exit("Run the `patch` command first.")
    code = ("import sys,torch;sys.path.insert(0,%r);"
            "from ViT_MoE import vit_base_patch16_224_in21k as f;"
            "m=f(pretrained=False,num_classes=2);"
            "n=sum(p.numel() for p in m.parameters() if p.requires_grad);"
            "y,_=m(torch.randn(2,3,224,224));"
            "print('CHECK_RESULT',n,tuple(m.head.weight.shape),tuple(y.shape))") % a.repo
    ok_all = True
    for mode in a.modes.split(","):
        env = dict(os.environ, HEAD_MODE=mode)
        out = subprocess.run([sys.executable, "-c", code], env=env, capture_output=True, text=True)
        line = [l for l in out.stdout.splitlines() if l.startswith("CHECK_RESULT")]
        if not line:
            print("[%s] FAILED to build the model:\n%s" % (mode, out.stderr[-1500:]))
            ok_all = False
            continue
        parts = line[0].split(None, 1)[1]
        n = int(parts.split()[0])
        want = EXPECTED[mode]
        good = (n == want) and ("(2, %d)" % HEAD_IN[mode]) in parts
        ok_all &= good
        print("[%s] trainable=%d expected=%d | head/logits=%s -> %s"
              % (mode, n, want, parts.split(None, 1)[1], "PASS" if good else "MISMATCH"))
    if not ok_all:
        sys.exit("Check failed - do not train until this is resolved.")
    print("All checks passed.")


def cmd_train(a):
    if not is_patched(a.repo):
        sys.exit("Run the `patch` command first.")
    os.makedirs(a.out, exist_ok=True)
    env = dict(os.environ, HEAD_MODE=a.mode)
    cmd = [sys.executable, "train.py", "--data_root", a.data_root, "--model_dir", a.out,
           "--epochs", str(a.epochs), "--batch_size", str(a.batch_size), "--seed", str(a.seed),
           "--num_workers", str(a.num_workers), "--device", str(a.device)]
    if a.resume >= 0:
        cmd += ["--resume", str(a.resume)]
    if a.extra:
        cmd += shlex.split(a.extra)
    cfg = {"mode": a.mode, "expected_trainable_params": EXPECTED[a.mode], "command": cmd,
           "repo_commit": git_hash(a.repo), "torch_timm": lib_versions(),
           "python": sys.version.split()[0], "started": time.strftime("%Y-%m-%d %H:%M:%S")}
    json.dump(cfg, open(os.path.join(a.out, "run_config.json"), "w"), indent=2)
    print("HEAD_MODE=%s  (log: %s/stdout_train.txt)" % (a.mode, a.out))
    print("Look for 'Trainable parameters: %d' in the first lines." % EXPECTED[a.mode])
    t0 = time.time()
    rc = run_tee(cmd, env, a.repo, os.path.join(a.out, "stdout_train.txt"))
    print("finished with exit code %d after %.1f min" % (rc, (time.time() - t0) / 60))
    sys.exit(rc)


def cmd_eval(a):
    if not is_patched(a.repo):
        sys.exit("Run the `patch` command first.")
    env = dict(os.environ, HEAD_MODE=a.mode)
    cmd = [sys.executable, a.eval_script, "--model_path", a.ckpt, "--test_root", a.test_root,
           "--dataset", a.dataset, "--frames", str(a.frames), "--num_workers", str(a.num_workers),
           "--device", str(a.device), "--out_json", a.out_json]
    if a.extra:
        cmd += shlex.split(a.extra)
    print("HEAD_MODE=%s - this must equal the mode the checkpoint was trained with." % a.mode)
    print("If eval.py rejects an option, run `python eval.py -h` in the repo and use --extra / "
          "--eval_script to match it.")
    rc = run_tee(cmd, env, a.repo, a.out_json + ".log")
    sys.exit(rc)


def _flatten(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(_flatten(v, prefix + str(k) + "."))
    else:
        out[prefix[:-1]] = d
    return out


def cmd_compare(a):
    fa, fb = _flatten(json.load(open(a.a))), _flatten(json.load(open(a.b)))
    keys = [k for k in fa if k in fb]
    print("%-40s %14s %14s %10s" % ("metric", a.a_name, a.b_name, "diff"))
    for k in keys:
        x, y = fa[k], fb[k]
        if isinstance(x, (int, float)) and isinstance(y, (int, float)):
            print("%-40s %14.5g %14.5g %+10.4g" % (k, x, y, y - x))
        else:
            print("%-40s %14s %14s" % (k, str(x)[:14], str(y)[:14]))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)

    p = sub.add_parser("patch"); p.add_argument("--repo", required=True); p.set_defaults(f=cmd_patch)
    p = sub.add_parser("unpatch"); p.add_argument("--repo", required=True); p.set_defaults(f=cmd_unpatch)

    p = sub.add_parser("check"); p.add_argument("--repo", required=True)
    p.add_argument("--modes", default="cls,multipool"); p.set_defaults(f=cmd_check)

    p = sub.add_parser("train")
    p.add_argument("--repo", required=True)
    p.add_argument("--data_root", required=True, help="folder containing train/ and valid/")
    p.add_argument("--out", required=True, help="output dir for checkpoints and logs")
    p.add_argument("--mode", default="multipool", choices=list(EXPECTED))
    p.add_argument("--epochs", type=int, default=5, help="must equal the baseline run (5)")
    p.add_argument("--batch_size", type=int, default=32)
    p.add_argument("--seed", type=int, default=2024)
    p.add_argument("--num_workers", type=int, default=4, help="use the same value as the baseline run")
    p.add_argument("--device", type=int, default=0)
    p.add_argument("--resume", type=int, default=-1, help="last finished epoch (0-based) to resume from")
    p.add_argument("--extra", default="", help="extra raw arguments for train.py")
    p.set_defaults(f=cmd_train)

    p = sub.add_parser("eval")
    p.add_argument("--repo", required=True)
    p.add_argument("--ckpt", required=True)
    p.add_argument("--test_root", required=True)
    p.add_argument("--dataset", default="FFPP", choices=["FFPP", "CDF", "DFDC_P"])
    p.add_argument("--frames", type=int, default=20)
    p.add_argument("--num_workers", type=int, default=2)
    p.add_argument("--device", type=int, default=0)
    p.add_argument("--out_json", required=True)
    p.add_argument("--mode", default="multipool", choices=list(EXPECTED),
                   help="use cls for the baseline checkpoint, multipool for the novelty")
    p.add_argument("--eval_script", default="eval.py")
    p.add_argument("--extra", default="")
    p.set_defaults(f=cmd_eval)

    p = sub.add_parser("compare")
    p.add_argument("--a", required=True, help="baseline result json")
    p.add_argument("--b", required=True, help="multipool result json")
    p.add_argument("--a_name", default="baseline")
    p.add_argument("--b_name", default="multipool")
    p.set_defaults(f=cmd_compare)

    a = ap.parse_args()
    a.f(a)


if __name__ == "__main__":
    main()


## 11. Patch the model and verify both architectures

In [ ]:
subprocess.check_call([sys.executable, SCRIPT, "patch", "--repo", REPO])
subprocess.check_call([sys.executable, SCRIPT, "check", "--repo", REPO])


**Expected:** `[cls] trainable=15510722 ... PASS` and `[multipool] trainable=15513794 ... PASS`, then `All checks passed.`
If not, **stop**.

## 12. Check the FF++ split folders and sample frames

In [ ]:
import glob
METHODS = ["original", "Deepfakes", "Face2Face", "FaceSwap", "NueralTextures"]
for sp in ["train", "valid", "test"]:
    row = {}
    for m in METHODS:
        d = glob.glob(f"{DATA_ROOT}/{sp}/{m}/*")
        row[m] = len([x for x in d if os.path.isdir(x)])
    print(sp, row)
print("Official FF++ split: train 720 per method, valid 140, test 140 (a subset is fine if the BASELINE used the same).")
one = glob.glob(f"{DATA_ROOT}/train/original/*/*.jpg")[:3]
print("sample frame files:", [os.path.basename(p) for p in one])

## 13. Download and verify the existing baseline checkpoint

In [ ]:
import urllib.request
os.makedirs(os.path.dirname(BASE_CKPT), exist_ok=True)
URL = "https://github.com/yagnaveenaaa/MoE-FFD/releases/download/baseline-epoch5-v1/models_params_4.tar"
WANT = "49432ca97fe6a07d72c9f02afcf1aa3bac8b386f91c6302ea152711ac399d6ad"
if not os.path.exists(BASE_CKPT):
    print("Downloading the published baseline checkpoint...")
    urllib.request.urlretrieve(URL, BASE_CKPT)
h = hashlib.sha256(open(BASE_CKPT, "rb").read()).hexdigest()
print("SHA-256:", h)
assert h == WANT, "Baseline checkpoint hash mismatch - do not continue"


## 14. Train the Multi-Pool model

This is intentionally a blocking foreground training command: it does not detach a background process, so notebook execution order is reliable and the next cell cannot race ahead of training. Do not execute this cell twice unless you intend to restart training.

In [ ]:
os.makedirs(OUT, exist_ok=True)
train_cmd = [
    sys.executable, SCRIPT, "train", "--repo", REPO, "--data_root", DATA_ROOT,
    "--out", OUT, "--mode", "multipool", "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE), "--seed", str(SEED),
    "--num_workers", str(NUM_WORKERS)
]
print("Starting:", " ".join(train_cmd))
with open(os.path.join(OUT, "notebook_console.txt"), "w") as log:
    proc = subprocess.Popen(train_cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
    rc = proc.wait()
assert rc == 0, f"Training failed with exit code {rc}; inspect {OUT}/notebook_console.txt and {OUT}/stdout_train.txt"
assert os.path.isfile(MP_CKPT), f"Expected checkpoint not found: {MP_CKPT}"
print("TRAINING COMPLETE:", MP_CKPT)


In [ ]:
print("Trainable parameter line:")
print(subprocess.run(["grep", "-h", "Trainable parameters", os.path.join(OUT, "stdout_train.txt")], capture_output=True, text=True).stdout)
print("Checkpoint exists:", os.path.isfile(MP_CKPT))
print("Output files:", sorted(os.listdir(OUT)))
assert os.path.isfile(MP_CKPT), "Multi-Pool checkpoint missing"


## 15. Evaluate baseline and Multi-Pool on the FF++ test split

In [ ]:
assert os.path.isfile(MP_CKPT), f"modified checkpoint not found: {MP_CKPT}"
subprocess.check_call([sys.executable, SCRIPT, "eval", "--repo", REPO, "--mode", "cls",
                       "--ckpt", BASE_CKPT, "--test_root", os.path.join(DATA_ROOT, "test"),
                       "--dataset", "FFPP", "--out_json", os.path.join(RES, "baseline_ffpp.json")])
subprocess.check_call([sys.executable, SCRIPT, "eval", "--repo", REPO, "--mode", "multipool",
                       "--ckpt", MP_CKPT, "--test_root", os.path.join(DATA_ROOT, "test"),
                       "--dataset", "FFPP", "--out_json", os.path.join(RES, "multipool_ffpp.json")])
subprocess.check_call([sys.executable, SCRIPT, "compare", "--a", os.path.join(RES, "baseline_ffpp.json"),
                       "--b", os.path.join(RES, "multipool_ffpp.json")])


## 16. Optional Celeb-DF evaluation

In [ ]:
if CDF_ROOT and os.path.isdir(CDF_ROOT):
    subprocess.check_call([sys.executable, SCRIPT, "eval", "--repo", REPO, "--mode", "cls",
                           "--ckpt", BASE_CKPT, "--test_root", CDF_ROOT, "--dataset", "CDF",
                           "--out_json", os.path.join(RES, "baseline_cdf.json")])
    subprocess.check_call([sys.executable, SCRIPT, "eval", "--repo", REPO, "--mode", "multipool",
                           "--ckpt", MP_CKPT, "--test_root", CDF_ROOT, "--dataset", "CDF",
                           "--out_json", os.path.join(RES, "multipool_cdf.json")])
    subprocess.check_call([sys.executable, SCRIPT, "compare", "--a", os.path.join(RES, "baseline_cdf.json"),
                           "--b", os.path.join(RES, "multipool_cdf.json")])
else:
    print("CDF_ROOT not set - skipped")


## 17. Final clean smoke-run and artifact audit

In [ ]:
# 1) Re-run architecture forward-pass/parameter-count checks.
subprocess.check_call([sys.executable, SCRIPT, "check", "--repo", REPO])

# 2) Confirm training and evaluation artifacts.
required = [BASE_CKPT, MP_CKPT,
            os.path.join(RES, "baseline_ffpp.json"),
            os.path.join(RES, "multipool_ffpp.json")]
missing = [p for p in required if not os.path.isfile(p)]
assert not missing, "Missing required final artifacts: " + repr(missing)

# 3) Parse result JSON files and ensure they are valid JSON objects.
for p in required[2:]:
    with open(p) as f:
        result = json.load(f)
    assert isinstance(result, (dict, list)), f"Unexpected result JSON type in {p}"
    print(os.path.basename(p), "JSON OK")

# 4) Confirm representative crops exist and are readable.
from PIL import Image
sample_candidates = glob.glob(os.path.join(DATA_ROOT, "train", "original", "*", "*.jpg"))
assert sample_candidates, "No readable crop candidate found in train/original"
with Image.open(sample_candidates[0]) as im:
    im.verify()
print("Sample crop:", sample_candidates[0], "| image verification: PASS")

# 5) Record a compact run manifest.
manifest = {
    "crop": {"frames_per_video": N_FRAMES, "face_detector": "MTCNN", "largest_face": True,
             "margin_fraction": MARGIN, "jpeg_quality": 95},
    "training": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "seed": SEED, "num_workers": NUM_WORKERS},
    "baseline_checkpoint": BASE_CKPT,
    "multipool_checkpoint": MP_CKPT,
    "results": [os.path.join(RES, "baseline_ffpp.json"), os.path.join(RES, "multipool_ffpp.json")],
    "smoke_test": "PASS"
}
with open(os.path.join(RES, "final_run_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)
print("\nFINAL SMOKE TEST: PASS")
print(json.dumps(manifest, indent=2))


## 18. Results to retain

The final outputs are in `/kaggle/working/results/` and `/kaggle/working/models/multipool/`. Preserve the notebook run with **Save Version**. The final smoke test must print `FINAL SMOKE TEST: PASS`.

**Reproducibility note:** The FF++ download and MTCNN crop stages depend on Kaggle Internet access, FF++ access approval, available disk space, and GPU/runtime availability. Those external conditions cannot be guaranteed by a notebook file alone. This notebook includes fail-fast checks so a failed download, missing folder, failed model patch, missing checkpoint, or invalid evaluation JSON is surfaced instead of being reported as a successful run.